In [4]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [5]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### MERIT DEM
#### Chosen for the project

In [ ]:
# Sentinel-2 cloud mask
def mask_s2_clouds(image):

    qa = image.select("QA60")

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return image.updateMask(mask).divide(10000)

# Sentinel-2 collection
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(panama_geom)
    .filterDate("2024-01-01", "2025-05-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .map(mask_s2_clouds)
)

# Composite clipped to Panama
image = s2.median().clip(panama_geom)

# Visualization parameters
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 0.3,
}

# Load DEM and clip to Panama, ~ 90m resolution
merit = ee.Image("MERIT/DEM/v1_0_3").clip(panama_geom)

# Elevation visualization
merit_vis = {
    "min": 0,
    "max": 3500,
    "palette": [
        "#000000",
        "#478fcd",
        "#86c58e",
        "#afc35e",
        "#8f7131",
        "#b78d4f",
        "#e2b8a6",
        "#ffffff"   # high mountains
    ],
}

# Add layers
Map.addLayer(image, rgb_vis, "Sentinel-2")
Map.addLayer(merit, merit_vis, "Elevation")

### NASA SRTM DEM
#### Alternative DEM #1

In [ ]:
# Sentinel-2 cloud mask
def mask_s2_clouds(image):

    qa = image.select("QA60")

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return image.updateMask(mask).divide(10000)

# Sentinel-2 collection
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(panama_geom)
    .filterDate("2024-01-01", "2025-05-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .map(mask_s2_clouds)
)

# Composite clipped to Panama
image = s2.median().clip(panama_geom)

# Visualization parameters
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 0.3,
}

# Load DEM and clip to Panama, ~ 30m resolution
srtm = ee.Image("USGS/SRTMGL1_003").clip(panama_geom)

# Create hillshade
hillshade = ee.Terrain.hillshade(srtm)

# Elevation visualization
dem_vis = {
    "min": 0,
    "max": 3500,
    "palette": [
        "#0b3d0b",  # dark green
        "#3f7f3f",
        "#b5a642",
        "#8c6d31",
        "#ffffff"   # high mountains
    ],
}

# Hillshade visualization
hillshade_vis = {
    "min": 0,
    "max": 255,
    "opacity": 0.4
}

# Add layers
Map.addLayer(image, rgb_vis, "Sentinel-2")
Map.addLayer(srtm, dem_vis, "Elevation")
Map.addLayer(hillshade, hillshade_vis, "Hillshade")

### Copernicus DEM
#### Alternative DEM #2

In [ ]:
# DEM collection, ~ 30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

# Create a single DEM image and clip it
copernicus = (
    dataset
    .select('DEM')
    #.mosaic()
    .first()
    .clip(panama_geom)
)

copernicus_vis = {
    'min': 0,
    'max': 2500,
    'palette': ['0000ff', '00ffff', 'ffff00', 'ff0000', 'ffffff'],
}

Map.addLayer(copernicus, copernicus_vis, 'Panama DEM')

In [ ]:
# Display map
Map.centerObject(panama_geom, 7)
Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…